In [7]:
from dotenv import load_dotenv
import os

# .envファイルの絶対パスを指定
env_path = os.path.join("..", ".env")  # 親ディレクトリの.envを指定
load_dotenv(env_path)

True

In [8]:
secret = os.getenv("SB_SECRET")
token = os.getenv('SB_TOKEN')

In [9]:
import time, uuid, hashlib, hmac, base64



def generate_headers():
    nonce = str(uuid.uuid4())
    t = str(int(round(time.time() * 1000)))
    string_to_sign = token + t + nonce
    sign = base64.b64encode(
        hmac.new(secret.encode(), string_to_sign.encode(), digestmod=hashlib.sha256).digest()
    ).decode()

    headers = {
        "Authorization": token,
        "sign": sign,
        "nonce": nonce,
        "t": t,
        "Content-Type": "application/json",
        "charset": "utf8"
    }
    return headers


In [10]:
import requests

url = "https://api.switch-bot.com/v1.1/devices"
response = requests.get(url, headers=generate_headers())
response.json()

{'statusCode': 100,
 'body': {'deviceList': [{'deviceId': '9C9E6EDCDB72',
    'deviceName': 'LED1',
    'deviceType': 'Color Bulb',
    'enableCloudService': True,
    'hubDeviceId': ''},
   {'deviceId': '9C9E6EDE6E06',
    'deviceName': 'LED2',
    'deviceType': 'Color Bulb',
    'enableCloudService': True,
    'hubDeviceId': ''},
   {'deviceId': '9C9E6EDE9D12',
    'deviceName': 'LED3',
    'deviceType': 'Color Bulb',
    'enableCloudService': True,
    'hubDeviceId': ''}],
  'infraredRemoteList': []},
 'message': 'success'}

In [11]:
def control_device(device_id, command, parameter="default", command_type="command"):
    url = f"https://api.switch-bot.com/v1.1/devices/{device_id}/commands"
    headers = generate_headers()
    body = {
        "command": command,           # 'turnOn' or 'turnOff'
        "parameter": parameter,
        "commandType": command_type
    }
    response = requests.post(url, json=body, headers=headers)
    print(f"[{device_id}] -> {command.upper()} -> {response.json()}")

In [20]:
# デバイス一覧
devices = {
    "LED1": "9C9E6EDCDB72",
    "LED2": "9C9E6EDE6E06",
    "LED3": "9C9E6EDE9D12"
}

# 全部ONにする
for name, device_id in devices.items():
    # 赤色に設定 (RGB形式: 255:0:0)
    # control_device(device_id, "setColor", "255:0:0")

    # # 明るさを80%に設定（範囲: 1〜100）
    # control_device(device_id, "setBrightness", "100")

    # 最後に点灯コマンド
    control_device(device_id, "turnOff")

# 全部OFFにする
# for name, device_id in devices.items():
#     control_device(device_id, "turnOff")


[9C9E6EDCDB72] -> TURNOFF -> {'statusCode': 100, 'body': {}, 'message': 'success'}
[9C9E6EDE6E06] -> TURNOFF -> {'statusCode': 100, 'body': {}, 'message': 'success'}
[9C9E6EDE9D12] -> TURNOFF -> {'statusCode': 100, 'body': {}, 'message': 'success'}


In [13]:
control_device(devices["LED3"], "turnOn")

[9C9E6EDE9D12] -> TURNON -> {'statusCode': 100, 'body': {}, 'message': 'success'}


In [16]:
url = f"https://api.switch-bot.com/v1.1/devices/9C9E6EDE9D12/commands"
headers = generate_headers()
body = {
"command": "turnOff",           # 'turnOn' or 'turnOff'
"parameter": "default",
"commandType": "command"
}
response = requests.post(url, json=body, headers=headers)
print(f"[9C9E6EDE9D12] -> turnOn -> {response.json()}")

[9C9E6EDE9D12] -> turnOn -> {'statusCode': 100, 'body': {}, 'message': 'success'}


In [15]:
import time
import uuid
import hmac
import hashlib
import base64
import aiohttp
import asyncio

class SwitchBotController:
    def __init__(self, token, secret):
        self.token = token
        self.secret = secret

    def _generate_headers(self):
        nonce = str(uuid.uuid4())
        t = str(int(round(time.time() * 1000)))
        string_to_sign = self.token + t + nonce
        sign = base64.b64encode(
            hmac.new(self.secret.encode(), string_to_sign.encode(), digestmod=hashlib.sha256).digest()
        ).decode()

        return {
            "Authorization": self.token,
            "sign": sign,
            "nonce": nonce,
            "t": t,
            "Content-Type": "application/json",
            "charset": "utf8"
        }

    async def _send_command(self, session, device_id, command, parameter="default", command_type="command"):
        url = f"https://api.switch-bot.com/v1.1/devices/{device_id}/commands"
        headers = self._generate_headers()
        payload = {
            "command": command,
            "parameter": parameter,
            "commandType": command_type
        }
        async with session.post(url, json=payload, headers=headers) as response:
            res_json = await response.json()
            print(f"[{device_id}] -> {command}({parameter}) -> {res_json}")
            return res_json
        

    async def turn_off(self, session, device_id: str):

        await self._send_command(session, device_id, command="turnOff")
       


    async def turn_on(self, session, device_id, rgb: tuple = (255,255,255), brightness: int = 100):
        r, g, b = rgb
        color_param = f"{r}:{g}:{b}"
        brightness_param = str(brightness)

        await self._send_command(session, device_id, "setColor", color_param)
        await asyncio.sleep(0.3)
        await self._send_command(session, device_id, "setBrightness", brightness_param)
        await asyncio.sleep(0.3)
        await self._send_command(session, device_id, "turnOn")


In [60]:
controller = SwitchBotController(token, secret)

In [1]:
import asyncio
import aiohttp

async def main():
    async with aiohttp.ClientSession() as session:
        tasks = []
        for device_id in devices.values():
            tasks.append(controller.turn_on(session, device_id, rgb=(255, 255, 0), brightness=80))
        await asyncio.gather(*tasks)

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop